# 02 — Vector Index with pgvector
Mirrors `examples/02_build_index_pgvector.py`.

In [ ]:
import os, json
from pathlib import Path
from packages.core.config import get_settings
from packages.retriever.embeddings import EmbeddingClient
print('Config (sanitised):', get_settings().as_sanitized_dict())

In [ ]:
from pathlib import Path
Path('data/samples').mkdir(parents=True, exist_ok=True)
Path('data/samples/acme.txt').write_text('ACME PLC Annual Report 2024: Gross margin was 38.2% for the fiscal year.', encoding='utf-8')
Path('data/samples/beta.txt').write_text('Beta Corp Q3 2024 revenue reached $1.26bn, up 14.5% YoY.', encoding='utf-8')
print('Seeded sample texts.')

In [ ]:
def chunk_text(text: str, max_chars=1200, overlap=120):
    s = ' '.join((text or '').replace('-\n','').replace('\n',' ').split())
    if not s: return []
    out=[]; i=0; step=max(1, max_chars-overlap)
    while i < len(s):
        out.append(s[i:i+max_chars]); i+=step
    return out

texts=[]
for f in sorted(Path('data/samples').glob('*.txt')):
    for i,ch in enumerate(chunk_text(f.read_text(encoding='utf-8', errors='ignore'))):
        texts.append({'id': f'{f.stem}#p{i}', 'doc_id': f.stem, 'text': ch, 'metadata': {'source': str(f)}})
len(texts)

In [ ]:
from packages.retriever.embeddings import EmbeddingClient
embedder = EmbeddingClient(provider='auto', model_alias='mini')
vecs = embedder.embed([t['text'] for t in texts], batch_size=64)
len(vecs), len(vecs[0])

In [ ]:
from packages.retriever.vectorstores.pgvector_store import PgVectorStore
cfg = get_settings()
collection = os.getenv('PGV_COLLECTION','finance_demo')
store = PgVectorStore(dsn=cfg.pgvector_url, collection=collection, create_if_missing=True)
B=128
for i in range(0, len(texts), B):
    batch = []
    for t,v in zip(texts[i:i+B], vecs[i:i+B]):
        batch.append({'id': t['id'], 'doc_id': t['doc_id'], 'text': t['text'], 'vector': v, 'metadata': t['metadata']})
    store.upsert(batch)
print('Upserted', len(texts), 'chunks into', collection)

In [ ]:
q = "What was ACME's gross margin in 2024?"
qv = embedder.embed([q])[0]
hits = store.search(qv, top_k=5)
[(round(h['score'],3), h['id']) for h in hits]